# Notebook 07B — Five-role V2 5-fold Kaggle Submission

**Inference-only notebook** for the Five-role 2.5D DINOv2 ViT-S/14 V2 ensemble trained in Notebook 07.

## Before running

1. Add the **RSNA Knee Abnormality Detection competition data** as Input.
2. Add the **Notebook Output from Notebook 07**, specifically the output containing `five_role_v2_cv5_submission_bundle.zip` (instructions and exact path below).
3. Enable a Kaggle GPU. **Keep Internet OFF** for the committed submission run, as required by the competition.
4. Run all cells. The final cell writes `/kaggle/working/submission.csv`.

### The one model artifact needed

Add only this generated bundle from Notebook 07:

`/kaggle/working/five_role_v2_cv5_submission_bundle.zip`

Inside it are the five `fold_*/best_checkpoint.pt` files and a local snapshot of the official DINOv2 source code. The complete backbone weights are already inside every model checkpoint, so this notebook creates the DINOv2 architecture with `pretrained=False` and does not download pretrained weights.

**Do not add** the 4.07-GiB `supplementary_five_role_cache`, CNN-LSTM shared labels/cache, `last_checkpoint.pt`, the single-split V1/V2 checkpoints, or OOF CSVs. Those are not needed for test inference.

The five fold models are ensembled by averaging their per-target probabilities. Test series are selected and decoded on the fly using the same five-role definitions, slice ordering, 12-slice sampling, percentile normalization, and 224×224 resizing used during training.

## 1. Imports and runtime

In [ ]:
import os
import re
import gc
import json
import math
import time
import copy
import shutil
import zipfile
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn

try:
    import cv2
    import pydicom
    try:
        from pydicom.pixels import apply_modality_lut
    except ImportError:
        from pydicom.pixel_data_handlers.util import apply_modality_lut
except ImportError as exc:
    raise RuntimeError(
        "The Kaggle image is missing pydicom or OpenCV. This submission is offline; "
        "use a Kaggle image/environment where these packages are preinstalled."
    ) from exc

warnings.filterwarnings("ignore", category=UserWarning)
SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)

if not torch.cuda.is_available():
    raise RuntimeError("GPU is required for this 5-model DINOv2 inference notebook. Enable a Kaggle GPU.")
device = torch.device("cuda")
USE_AMP = True

print("Python:", __import__("sys").version.split()[0])
print("PyTorch:", torch.__version__)
print("pydicom:", pydicom.__version__)
print("OpenCV:", cv2.__version__)
print("GPU:", torch.cuda.get_device_name(0))
print("GPU memory GiB:", round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 2))
print("Internet must remain OFF for the committed Kaggle submission run.")

## 2. Competition test data and submission schema

In [ ]:
INPUT_ROOT = Path("/kaggle/input")
COMP_ROOT_OVERRIDE = ""  # Optional exact directory containing test.csv.

if COMP_ROOT_OVERRIDE:
    COMP_ROOT = Path(COMP_ROOT_OVERRIDE)
else:
    defaults = [INPUT_ROOT / "competitions" / "rsna-knee-abnormality-detection"]
    competitions_root = INPUT_ROOT / "competitions"
    if competitions_root.is_dir():
        defaults += sorted(competitions_root.glob("*"))
    COMP_ROOT = next((p for p in defaults if (p / "test.csv").is_file()), None)
    if COMP_ROOT is None:
        matches = []
        for root, dirs, files in os.walk(INPUT_ROOT):
            dirs[:] = [d for d in dirs if d.lower() not in {"train_series", "test_series", "train_images", "test_images", ".git"}]
            if "test.csv" in files and "sample_submission.csv" in files:
                matches.append(Path(root))
            if len(matches) > 5:
                break
        if len(matches) != 1:
            raise FileNotFoundError(f"Could not uniquely locate competition test.csv. Candidates: {matches}")
        COMP_ROOT = matches[0]

TEST_CSV = COMP_ROOT / "test.csv"
TEST_SERIES_CSV = COMP_ROOT / "test_series.csv"
SAMPLE_SUBMISSION_CSV = COMP_ROOT / "sample_submission.csv"
TEST_SERIES_DIR = next(
    (p for p in [COMP_ROOT / "test_series", COMP_ROOT / "test_images", COMP_ROOT / "test_dicom"] if p.is_dir()),
    None,
)
for path in [TEST_CSV, TEST_SERIES_CSV, SAMPLE_SUBMISSION_CSV]:
    if not path.is_file():
        raise FileNotFoundError(path)
if TEST_SERIES_DIR is None:
    raise FileNotFoundError("Could not find test_series/ DICOM directory under the competition input.")

ID_COL = "StudyInstanceUID"
TARGETS = [
    "ACL", "MCL", "Medial Meniscus", "Lateral Meniscus",
    "Medial OA", "Lateral OA", "PF OA", "Effusion",
    "Synovitis", "Baker's", "Contusion", "Fracture",
]
EXPECTED_COLUMNS = [ID_COL, *TARGETS]

test_df = pd.read_csv(TEST_CSV, dtype={ID_COL: str})
test_series_df = pd.read_csv(TEST_SERIES_CSV, dtype={ID_COL: str, "SeriesInstanceUID": str})
sample_submission = pd.read_csv(SAMPLE_SUBMISSION_CSV, dtype={ID_COL: str})
for frame in [test_df, test_series_df, sample_submission]:
    frame[ID_COL] = frame[ID_COL].astype(str)

if sample_submission.columns.tolist() != EXPECTED_COLUMNS:
    raise ValueError(
        "sample_submission schema/target order mismatch.\n"
        f"Expected: {EXPECTED_COLUMNS}\nGot: {sample_submission.columns.tolist()}"
    )
if test_df[ID_COL].duplicated().any() or sample_submission[ID_COL].duplicated().any():
    raise ValueError("Duplicate StudyInstanceUID in test.csv or sample_submission.csv.")
if set(test_df[ID_COL]) != set(sample_submission[ID_COL]):
    raise ValueError("test.csv and sample_submission.csv contain different study UIDs.")
required_series_cols = {ID_COL, "SeriesInstanceUID", "Anatomical_Plane", "Fluid_Sensitive", "Fat_Suppression"}
missing_cols = required_series_cols - set(test_series_df.columns)
if missing_cols:
    raise ValueError(f"test_series.csv missing columns: {sorted(missing_cols)}")

OUTPUT_ROOT = Path("/kaggle/working/five_role_v2_submission")
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

print("Competition root:", COMP_ROOT)
print("Test DICOM root:", TEST_SERIES_DIR)
print("Test studies:", len(test_df))
print("Test series rows:", len(test_series_df))
print("Submission rows:", len(sample_submission))
print("Visible sample test may contain only 3 studies; scoring replaces it with the full hidden test set.")

## 3. Locate and unpack the single model bundle

In [ ]:
# ============================================================
# CELL 6 — Locate and unpack the five-fold model bundle
# Replace the ENTIRE previous Cell 6 with this code.
# Finds the artifact by CONTENT, not Kaggle Dataset slug or ZIP filename.
# ============================================================

import os
import json
import shutil
import zipfile
from pathlib import Path, PurePosixPath

INPUT_ROOT = Path('/kaggle/input')
OUTPUT_ROOT = Path('/kaggle/working/five_role_v2_submission')
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

# Optional: set this only if automatic content-based discovery finds more than one bundle.
# It can point to either a ZIP file or an extracted bundle root.
BUNDLE_PATH_OVERRIDE = ''


def _is_complete_bundle_root(root: Path) -> bool:
    """Validate a bundle root using all required files."""
    if not (root / 'bundle_manifest.json').is_file():
        return False
    if not (root / 'dinov2_repo' / 'hubconf.py').is_file():
        return False
    if not (root / 'dinov2_repo' / 'dinov2').is_dir():
        return False
    return all(
        (root / 'five_role_2p5d_v2_cv5' / f'fold_{fold}' / 'best_checkpoint.pt').is_file()
        for fold in range(1, 6)
    )


def _zip_contains_complete_bundle(zip_path: Path) -> bool:
    """Inspect ZIP central directory; do not extract unrelated archives."""
    try:
        with zipfile.ZipFile(zip_path, 'r') as zf:
            names = {
                name.replace('\\', '/').lstrip('./').rstrip('/')
                for name in zf.namelist()
            }
        def has_suffix(suffix: str) -> bool:
            return any(name.endswith(suffix) for name in names)
        required = [
            'bundle_manifest.json',
            'dinov2_repo/hubconf.py',
            'five_role_2p5d_v2_cv5/fold_1/best_checkpoint.pt',
            'five_role_2p5d_v2_cv5/fold_2/best_checkpoint.pt',
            'five_role_2p5d_v2_cv5/fold_3/best_checkpoint.pt',
            'five_role_2p5d_v2_cv5/fold_4/best_checkpoint.pt',
            'five_role_2p5d_v2_cv5/fold_5/best_checkpoint.pt',
        ]
        return all(has_suffix(item) for item in required)
    except (zipfile.BadZipFile, OSError, PermissionError):
        return False


# Discover candidates. Skip DICOM trees because they contain many files and cannot
# contain the model bundle we are looking for.
skip_dir_names = {
    'train_series', 'test_series', 'train_images', 'test_images',
    'train_dicom', 'test_dicom', '.git', '__pycache__',
}
zip_candidates = []
extracted_candidates = []

if BUNDLE_PATH_OVERRIDE:
    override = Path(BUNDLE_PATH_OVERRIDE)
    if override.is_file() and override.suffix.lower() == '.zip':
        if not _zip_contains_complete_bundle(override):
            raise FileNotFoundError(
                f'The override ZIP does not contain the expected five-fold bundle: {override}'
            )
        zip_candidates = [override]
    elif override.is_dir() and _is_complete_bundle_root(override):
        extracted_candidates = [override]
    else:
        raise FileNotFoundError(
            'BUNDLE_PATH_OVERRIDE must point to the correct ZIP or extracted bundle root. '
            f'Received: {override}'
        )
else:
    for current_root, dirs, files in os.walk(INPUT_ROOT):
        dirs[:] = [d for d in dirs if d.lower() not in skip_dir_names]
        current = Path(current_root)

        # Check whether this directory itself is an extracted bundle root.
        if 'bundle_manifest.json' in files and _is_complete_bundle_root(current):
            extracted_candidates.append(current)

        # Check every ZIP by its internal contents, regardless of its external filename.
        for filename in files:
            candidate = current / filename
            if candidate.suffix.lower() == '.zip' and _zip_contains_complete_bundle(candidate):
                zip_candidates.append(candidate)

# Deduplicate by resolved path.
zip_candidates = list({str(p.resolve()): p for p in zip_candidates}.values())
extracted_candidates = list({str(p.resolve()): p for p in extracted_candidates}.values())

if len(zip_candidates) == 1:
    selected_zip = zip_candidates[0]
    selected_extracted = None
elif len(zip_candidates) == 0 and len(extracted_candidates) == 1:
    selected_zip = None
    selected_extracted = extracted_candidates[0]
else:
    print('========== BUNDLE DISCOVERY DEBUG ==========')
    print('INPUT_ROOT:', INPUT_ROOT)
    print('Top-level Kaggle inputs:')
    if INPUT_ROOT.is_dir():
        for p in sorted(INPUT_ROOT.iterdir()):
            print(' -', '[DIR]' if p.is_dir() else '[FILE]', p)
    print('Matching ZIP bundles:', [str(p) for p in zip_candidates])
    print('Complete extracted bundles:', [str(p) for p in extracted_candidates])
    raise FileNotFoundError(
        'Expected exactly one five-role V2 CV5 bundle by internal content. '
        'The debug listing above shows what Kaggle actually mounted. '
        'No training or model files were modified.'
    )

# Extract the chosen ZIP into writable working storage, or use the read-only input folder.
if selected_zip is not None:
    BUNDLE_PATH = selected_zip
    EXTRACT_ROOT = OUTPUT_ROOT / 'model_bundle'
    if EXTRACT_ROOT.exists():
        shutil.rmtree(EXTRACT_ROOT)
    EXTRACT_ROOT.mkdir(parents=True, exist_ok=True)

    with zipfile.ZipFile(BUNDLE_PATH, 'r') as zf:
        bad_member = zf.testzip()
        if bad_member is not None:
            raise RuntimeError(f'Model bundle ZIP is corrupt at member: {bad_member}')
        for member in zf.namelist():
            normalized = member.replace('\\', '/')
            member_path = PurePosixPath(normalized)
            if normalized.startswith('/') or '..' in member_path.parts:
                raise RuntimeError(f'Unsafe path inside model bundle: {member}')
        zf.extractall(EXTRACT_ROOT)

    # ZIPs can contain one extra enclosing directory; locate the actual bundle root.
    extracted_roots = [
        p.parent for p in EXTRACT_ROOT.rglob('bundle_manifest.json')
        if _is_complete_bundle_root(p.parent)
    ]
    extracted_roots = list({str(p.resolve()): p for p in extracted_roots}.values())
    if len(extracted_roots) != 1:
        raise FileNotFoundError(
            'ZIP was found and extracted, but its internal bundle root is not unique/complete. '
            f'Valid roots found: {[str(p) for p in extracted_roots]}'
        )
    EXTRACT_ROOT = extracted_roots[0]
else:
    BUNDLE_PATH = selected_extracted
    EXTRACT_ROOT = selected_extracted

# Define variables used by the remaining submission notebook cells.
DINOV2_REPO_DIR = EXTRACT_ROOT / 'dinov2_repo'
CV_ROOT = EXTRACT_ROOT / 'five_role_2p5d_v2_cv5'
BUNDLE_MANIFEST_PATH = EXTRACT_ROOT / 'bundle_manifest.json'
FOLD_CKPTS = [
    CV_ROOT / f'fold_{fold}' / 'best_checkpoint.pt'
    for fold in range(1, 6)
]

bundle_manifest = json.loads(BUNDLE_MANIFEST_PATH.read_text(encoding='utf-8'))
for fold, checkpoint_path in enumerate(FOLD_CKPTS, start=1):
    if not checkpoint_path.is_file():
        raise FileNotFoundError(f'Missing fold {fold} best checkpoint: {checkpoint_path}')

print('========== MODEL BUNDLE LOCATED SUCCESSFULLY ==========')
print('Bundle source:', BUNDLE_PATH)
print('Bundle type:', 'ZIP archive' if selected_zip is not None else 'extracted Kaggle input folder')
print('Resolved bundle root:', EXTRACT_ROOT)
print('DINOv2 source:', DINOV2_REPO_DIR)
print('Best checkpoints:')
for checkpoint_path in FOLD_CKPTS:
    print(' -', checkpoint_path)
print('Architecture:', bundle_manifest.get('architecture'))
print('All five fold checkpoints and DINOv2 source are present.')


## 4. Define the five test roles and choose one series per role

In [ ]:
IMAGE_SIZE = 224
PATCH_SIZE = 14
BACKBONE_DIM = 384
EMBED_DIM = 256
FREEZE_EARLY_BLOCKS = 9
N_ROLE_SLICES = 12
N_ROLE_WINDOWS = N_ROLE_SLICES - 2
FIVE_ROLE_ORDER = [
    "SAG_FLUID", "COR_FLUID", "AX_FLUID", "SAG_NONFLUID", "COR_NONFLUID"
]
DINO_HUB_MODEL = "dinov2_vits14"
ENCODER_CHUNK_SIZE = 8
MIN_DECODED_SLICES = 4

ROLE_SPECS = {
    "SAG_FLUID": ("sagittal", 1),
    "COR_FLUID": ("coronal", 1),
    "AX_FLUID": ("axial", 1),
    "SAG_NONFLUID": ("sagittal", 0),
    "COR_NONFLUID": ("coronal", 0),
}
PLANES = ["sagittal", "coronal", "axial"]

def normalize_plane(value):
    s = str(value).strip().lower()
    if "sag" in s:
        return "sagittal"
    if "cor" in s:
        return "coronal"
    if "axi" in s or "trans" in s or s == "ax":
        return "axial"
    return s

def parse_binary_flag(value):
    if pd.isna(value):
        return -1
    s = str(value).strip().lower()
    mapping = {
        "true": 1, "false": 0, "yes": 1, "no": 0,
        "y": 1, "n": 0, "1": 1, "0": 0,
        "1.0": 1, "0.0": 0,
    }
    if s in mapping:
        return mapping[s]
    try:
        x = float(s)
        return int(x) if x in (0.0, 1.0) else -1
    except Exception:
        return -1

test_series_df["plane_norm"] = test_series_df["Anatomical_Plane"].map(normalize_plane)
test_series_df["fluid_num"] = test_series_df["Fluid_Sensitive"].map(parse_binary_flag)
test_series_df["fat_num"] = test_series_df["Fat_Suppression"].map(parse_binary_flag)
test_series_df["SeriesInstanceUID"] = test_series_df["SeriesInstanceUID"].astype(str)

# Reproduce the two-stage selection used while building the training five-role index:
# 1) The original CNN-LSTM cache chose one primary series per plane by
#    Fluid_Sensitive DESC, Fat_Suppression DESC, SeriesInstanceUID ASC.
# 2) If that primary series did not match a role, Cell C selected a separate
#    fallback from the exact plane/fluid candidates with Fat_Suppression=0,
#    then unknown, then 1.
role_selection = {}
selection_rows = []
series_groups = {str(uid): group.copy() for uid, group in test_series_df.groupby(ID_COL, sort=False)}

for uid in test_df[ID_COL].astype(str).tolist():
    group = series_groups.get(uid, test_series_df.iloc[0:0])
    role_selection[uid] = {}
    primary_by_plane = {}

    for plane in PLANES:
        primary_candidates = group[group["plane_norm"].eq(plane)].copy()
        if primary_candidates.empty:
            primary_by_plane[plane] = None
        else:
            primary_candidates = primary_candidates.sort_values(
                ["fluid_num", "fat_num", "SeriesInstanceUID"],
                ascending=[False, False, True], kind="mergesort"
            )
            primary_by_plane[plane] = primary_candidates.iloc[0]

    for role in FIVE_ROLE_ORDER:
        plane, fluid_flag = ROLE_SPECS[role]
        candidates = group[
            group["plane_norm"].eq(plane) & group["fluid_num"].eq(fluid_flag)
        ].copy()
        candidate_count = len(candidates)
        primary = primary_by_plane.get(plane)
        selected_uid = None
        selection_source = "missing"

        if primary is not None and int(primary["fluid_num"]) == fluid_flag:
            selected_uid = str(primary["SeriesInstanceUID"])
            selection_source = "primary_cache_equivalent"
        elif not candidates.empty:
            candidates["_fat_rank"] = candidates["fat_num"].map({0: 0, -1: 1, 1: 2}).fillna(1)
            candidates = candidates.sort_values(
                ["_fat_rank", "SeriesInstanceUID"], ascending=[True, True], kind="mergesort"
            )
            selected_uid = str(candidates.iloc[0]["SeriesInstanceUID"])
            selection_source = "supplementary_fallback"

        role_selection[uid][role] = selected_uid
        selection_rows.append({
            ID_COL: uid,
            "role": role,
            "plane": plane,
            "fluid_flag": fluid_flag,
            "candidate_count": int(candidate_count),
            "primary_series_uid": str(primary["SeriesInstanceUID"]) if primary is not None else "",
            "SeriesInstanceUID": selected_uid or "",
            "selection_source": selection_source,
            "selection_status": "selected" if selected_uid else "no_candidate",
        })

role_selection_df = pd.DataFrame(selection_rows)
role_selection_df.to_csv(OUTPUT_ROOT / "test_role_selection.csv", index=False)
print("========== TEST ROLE SELECTION ==========")
display(role_selection_df.groupby(["role", "selection_source"]).size().rename("n_studies").reset_index())
print("Selected role series rows:", int(role_selection_df["SeriesInstanceUID"].ne("").sum()))

## 5. DICOM decoding and training-matched slice preprocessing

In [ ]:
def safe_float(value, default=np.nan):
    try:
        return float(value)
    except Exception:
        return default

def natural_key(name):
    return [int(part) if part.isdigit() else part.lower() for part in re.split(r"(\d+)", name)]

DICOM_HEADER_TAGS = [
    "ImagePositionPatient", "ImageOrientationPatient", "InstanceNumber", "SOPInstanceUID",
    "PixelSpacing", "Rows", "Columns", "SeriesDescription", "ProtocolName", "SequenceName",
    "Laterality", "ImageLaterality", "FrameOfReferenceUID", "SliceThickness",
    "SpacingBetweenSlices", "PhotometricInterpretation", "RescaleSlope", "RescaleIntercept",
    "StudyInstanceUID", "SeriesInstanceUID",
]

def read_sorted_dicom_headers(series_dir: Path):
    files = sorted(p for p in series_dir.iterdir() if p.is_file() and p.suffix.lower() == ".dcm")
    if not files:
        files = sorted(p for p in series_dir.iterdir() if p.is_file() and not p.name.startswith("."))
    rows = []
    first_normal = None
    for path in files:
        try:
            ds = pydicom.dcmread(
                str(path), stop_before_pixels=True, force=True, specific_tags=DICOM_HEADER_TAGS
            )
            iop = getattr(ds, "ImageOrientationPatient", None)
            ipp = getattr(ds, "ImagePositionPatient", None)
            pos = np.nan
            instance = safe_float(getattr(ds, "InstanceNumber", np.nan))
            if iop is not None and ipp is not None and len(iop) >= 6 and len(ipp) >= 3:
                normal = np.cross(np.asarray(iop[:3], dtype=np.float64), np.asarray(iop[3:6], dtype=np.float64))
                norm = np.linalg.norm(normal)
                if norm > 0:
                    normal = normal / norm
                    if first_normal is None:
                        first_normal = normal
                    pos = float(np.dot(np.asarray(ipp[:3], dtype=np.float64), first_normal))
            rows.append({
                "path": path, "filename": path.name, "position_mm": pos,
                "instance": instance, "sop_uid": str(getattr(ds, "SOPInstanceUID", "")),
            })
        except Exception as exc:
            rows.append({
                "path": path, "filename": path.name, "position_mm": np.nan,
                "instance": np.nan, "sop_uid": "", "header_error": str(exc),
            })
    if not rows:
        return [], "no_dicom_files"
    n = len(rows)
    npos = sum(np.isfinite(r["position_mm"]) for r in rows)
    ninst = sum(np.isfinite(r["instance"]) for r in rows)
    if npos >= max(2, int(0.7 * n)):
        rows.sort(key=lambda r: (np.inf if not np.isfinite(r["position_mm"]) else r["position_mm"], r["filename"]))
        mode = "ImagePositionPatient_projected"
    elif ninst >= max(2, int(0.7 * n)):
        rows.sort(key=lambda r: (np.inf if not np.isfinite(r["instance"]) else r["instance"], r["filename"]))
        mode = "InstanceNumber"
    else:
        rows.sort(key=lambda r: natural_key(r["filename"]))
        mode = "filename_fallback"
    for i, row in enumerate(rows):
        row["sort_index"] = i
    return rows, mode

def decode_dicom_slice(path: Path):
    ds = pydicom.dcmread(str(path), force=True)
    pixels = np.asarray(ds.pixel_array)
    if pixels.ndim == 3:
        if pixels.shape[-1] in (3, 4):
            pixels = pixels[..., :3].astype(np.float32)
            pixels = 0.299 * pixels[..., 0] + 0.587 * pixels[..., 1] + 0.114 * pixels[..., 2]
        else:
            pixels = pixels[pixels.shape[0] // 2]
    pixels = np.squeeze(pixels)
    if pixels.ndim != 2:
        raise ValueError(f"Expected 2D MRI slice, got shape={pixels.shape}")
    pixels = pixels.astype(np.float32)
    try:
        pixels = np.asarray(apply_modality_lut(pixels, ds), dtype=np.float32)
    except Exception:
        slope = safe_float(getattr(ds, "RescaleSlope", 1.0), 1.0)
        intercept = safe_float(getattr(ds, "RescaleIntercept", 0.0), 0.0)
        pixels = pixels * slope + intercept
    if str(getattr(ds, "PhotometricInterpretation", "MONOCHROME2")).upper() == "MONOCHROME1":
        lo, hi = np.nanmin(pixels), np.nanmax(pixels)
        pixels = hi + lo - pixels
    pixels[~np.isfinite(pixels)] = 0
    return pixels, ds

def process_role_series(study_uid: str, series_uid: str):
    # Decode one selected series into 12 uint8 224x224 slices, matching training Cell D.
    zero_images = np.zeros((N_ROLE_SLICES, IMAGE_SIZE, IMAGE_SIZE), dtype=np.uint8)
    zero_valid = np.zeros(N_ROLE_SLICES, dtype=np.uint8)
    zero_pos = np.zeros(N_ROLE_SLICES, dtype=np.float32)
    series_dir = TEST_SERIES_DIR / str(study_uid) / str(series_uid)
    if not series_dir.is_dir():
        return None, None, None, {"status": "missing_series_directory", "n_files": 0, "n_decoded": 0}
    headers, sort_mode = read_sorted_dicom_headers(series_dir)
    if not headers:
        return None, None, None, {"status": sort_mode, "n_files": 0, "n_decoded": 0}

    n = len(headers)
    initial = np.unique(np.linspace(0, n - 1, min(N_ROLE_SLICES, n)).round().astype(int)).tolist()
    initial_set = set(initial)
    candidates = initial + [i for i in range(n) if i not in initial_set]
    decoded_images, decoded_positions, errors = [], [], []
    for header_idx in candidates:
        header = headers[header_idx]
        try:
            image, ds = decode_dicom_slice(header["path"])
            actual_study = str(getattr(ds, "StudyInstanceUID", ""))
            actual_series = str(getattr(ds, "SeriesInstanceUID", ""))
            if actual_study and actual_study != str(study_uid):
                raise ValueError("StudyInstanceUID mismatch")
            if actual_series and actual_series != str(series_uid):
                raise ValueError("SeriesInstanceUID mismatch")
            decoded_images.append(image)
            decoded_positions.append(header["sort_index"] / max(n - 1, 1))
            if len(decoded_images) >= N_ROLE_SLICES:
                break
        except Exception as exc:
            errors.append(f"{header['filename']}:{type(exc).__name__}:{exc}")

    if len(decoded_images) < MIN_DECODED_SLICES:
        return None, None, None, {
            "status": "too_few_decoded_slices",
            "n_files": n,
            "n_decoded": len(decoded_images),
            "sort_mode": sort_mode,
            "error": " | ".join(errors[:2]),
        }

    order = np.argsort(decoded_positions, kind="stable")
    decoded_images = [decoded_images[i] for i in order]
    decoded_positions = [decoded_positions[i] for i in order]
    values = np.concatenate([image.reshape(-1) for image in decoded_images])
    lo, hi = np.percentile(values, [1.0, 99.5])
    if not np.isfinite(lo) or not np.isfinite(hi) or hi <= lo:
        lo, hi = float(values.min()), float(values.max())
    if hi <= lo:
        hi = lo + 1.0

    out_images = zero_images.copy()
    out_valid = zero_valid.copy()
    out_pos = zero_pos.copy()
    for slot, (image, position) in enumerate(zip(decoded_images, decoded_positions)):
        x = np.clip((np.clip(image, lo, hi) - lo) / (hi - lo), 0, 1)
        x = (x * 255.0).round().astype(np.uint8)
        out_images[slot] = cv2.resize(x, (IMAGE_SIZE, IMAGE_SIZE), interpolation=cv2.INTER_AREA)
        out_valid[slot] = 1
        out_pos[slot] = float(position)
    return out_images, out_valid, out_pos, {
        "status": "ok", "n_files": n, "n_decoded": len(decoded_images),
        "sort_mode": sort_mode, "error": " | ".join(errors[:3]),
    }

print("DICOM preprocessing functions ready.")

## 6. Build one study as five-role 2.5D input

In [ ]:
def build_study_windows(study_uid: str):
    # Returns X [5,10,3,224,224], valid [5,10], positions [5,10], and diagnostics.
    X = np.zeros(
        (len(FIVE_ROLE_ORDER), N_ROLE_WINDOWS, 3, IMAGE_SIZE, IMAGE_SIZE), dtype=np.uint8
    )
    window_valid = np.zeros((len(FIVE_ROLE_ORDER), N_ROLE_WINDOWS), dtype=np.uint8)
    window_positions = np.zeros((len(FIVE_ROLE_ORDER), N_ROLE_WINDOWS), dtype=np.float32)
    diagnostics = []

    for role_idx, role in enumerate(FIVE_ROLE_ORDER):
        series_uid = role_selection.get(str(study_uid), {}).get(role)
        if not series_uid:
            diagnostics.append({
                ID_COL: str(study_uid), "role": role, "SeriesInstanceUID": "",
                "status": "no_candidate_series", "n_files": 0, "n_decoded": 0,
            })
            continue
        slices, slice_valid, slice_pos, info = process_role_series(study_uid, series_uid)
        diagnostics.append({
            ID_COL: str(study_uid), "role": role, "SeriesInstanceUID": str(series_uid), **info
        })
        if slices is None:
            continue
        for window_idx in range(N_ROLE_WINDOWS):
            local_valid = slice_valid[window_idx:window_idx + 3]
            if int(local_valid.sum()) >= 2:
                X[role_idx, window_idx] = slices[window_idx:window_idx + 3]
                window_valid[role_idx, window_idx] = 1
                window_positions[role_idx, window_idx] = float(
                    slice_pos[window_idx:window_idx + 3][local_valid.astype(bool)].mean()
                )

    n_valid_windows = int(window_valid.sum())
    n_available_roles = int(window_valid.any(axis=1).sum())
    if n_valid_windows == 0:
        return None, None, None, diagnostics, n_available_roles
    return X, window_valid, window_positions, diagnostics, n_available_roles

print("One-study window builder ready.")
print("Expected model input: images [5,10,3,224,224], mask/positions [5,10].")

## 7. Define the exact Five-role V2 architecture

In [ ]:
# ============================================================
# CELL I — Five-role 2.5D DINOv2 model (V2: target x role pooling)
# Shared ViT-S/14 encoder; each target pools each role independently;
# five role vectors are concatenated and sent to a target-specific head.
# ============================================================
import math
import torch
import torch.nn as nn


class FiveRoleDINOv2_2p5D(nn.Module):
    def __init__(
        self,
        backbone,
        image_size=IMAGE_SIZE,
        backbone_dim=BACKBONE_DIM,
        embed_dim=EMBED_DIM,
        freeze_early_blocks=FREEZE_EARLY_BLOCKS,
        dropout=0.15,
        encoder_chunk_size=8,
    ):
        super().__init__()
        self.backbone = backbone
        self.image_size = int(image_size)
        self.backbone_dim = int(backbone_dim)
        self.embed_dim = int(embed_dim)
        self.encoder_chunk_size = int(encoder_chunk_size)
        self.n_roles = len(FIVE_ROLE_ORDER)
        self.n_targets = len(TARGETS)
        self.n_windows = N_ROLE_WINDOWS

        if self.image_size % PATCH_SIZE != 0:
            raise ValueError("Image size must be divisible by DINOv2 patch size 14.")
        if not hasattr(self.backbone, "blocks") or len(self.backbone.blocks) != 12:
            raise ValueError("Expected DINOv2 ViT-S/14 with 12 transformer blocks.")
        if self.n_roles != 5:
            raise ValueError(f"Expected exactly five roles, found {self.n_roles}.")

        # Preserve the existing fine-tuning policy: freeze the backbone first,
        # then unfreeze its last blocks and final LayerNorm.
        for p in self.backbone.parameters():
            p.requires_grad_(False)
        if not 0 <= freeze_early_blocks < 12:
            raise ValueError("freeze_early_blocks must be between 0 and 11.")
        self.trainable_blocks = list(self.backbone.blocks[freeze_early_blocks:])
        for block in self.trainable_blocks:
            for p in block.parameters():
                p.requires_grad_(True)
        for p in self.backbone.norm.parameters():
            p.requires_grad_(True)

        # Convert each DINOv2 window into one compact feature vector.
        self.patch_attention = nn.Sequential(
            nn.LayerNorm(self.backbone_dim),
            nn.Linear(self.backbone_dim, 1, bias=False),
        )
        self.window_projection = nn.Sequential(
            nn.Linear(self.backbone_dim * 2, self.embed_dim),
            nn.LayerNorm(self.embed_dim),
            nn.GELU(),
            nn.Dropout(dropout),
        )

        # Mild spatial prior; peripheral patches remain available to attention.
        grid = self.image_size // PATCH_SIZE
        coords = torch.linspace(-1.0, 1.0, grid)
        yy, xx = torch.meshgrid(coords, coords, indexing="ij")
        log_gaussian = -0.5 * ((xx / 0.55) ** 2 + (yy / 0.55) ** 2)
        log_gaussian = log_gaussian - log_gaussian.max()
        self.register_buffer("center_bias", (0.15 * log_gaussian.reshape(-1)).float())

        self.role_embedding = nn.Parameter(
            torch.zeros(1, self.n_roles, 1, self.embed_dim)
        )
        self.window_embedding = nn.Embedding(self.n_windows, self.embed_dim)
        self.position_projection = nn.Sequential(
            nn.Linear(3, self.embed_dim),
            nn.GELU(),
            nn.Linear(self.embed_dim, self.embed_dim),
        )
        nn.init.normal_(self.role_embedding, std=0.02)

        # V2: independent attention query for every (target, role) pair.
        # Shape: [12 targets, 5 roles, embedding dimension].
        self.target_role_queries = nn.Parameter(
            torch.empty(self.n_targets, self.n_roles, self.embed_dim)
        )
        nn.init.normal_(self.target_role_queries, std=0.5)

        # h(target, role) = global_role + gamma * (attended_role - global_role).
        # Start gamma at 0.70, not 0.10, so target-specific evidence is not
        # almost suppressed at initialization.
        gamma_init = 0.70
        gamma_logit = math.log(gamma_init / (1.0 - gamma_init))
        self.target_role_residual_logits = nn.Parameter(
            torch.full((self.n_targets, self.n_roles), gamma_logit)
        )
        self.role_feature_norm = nn.LayerNorm(self.embed_dim)

        # Every target gets its own prediction head. Each head sees the five
        # role-specific vectors concatenated in FIVE_ROLE_ORDER, plus presence
        # indicators so a missing role is distinguishable from a real zero.
        head_input_dim = self.n_roles * self.embed_dim + self.n_roles
        self.target_classifiers = nn.ModuleList([
            nn.Sequential(
                nn.LayerNorm(head_input_dim),
                nn.Linear(head_input_dim, 128),
                nn.GELU(),
                nn.Dropout(0.25),
                nn.Linear(128, 1),
            )
            for _ in range(self.n_targets)
        ])

        mean = torch.tensor([0.485, 0.456, 0.406]).view(1, 3, 1, 1)
        std = torch.tensor([0.229, 0.224, 0.225]).view(1, 3, 1, 1)
        self.register_buffer("image_mean", mean)
        self.register_buffer("image_std", std)

    def train(self, mode=True):
        super().train(mode)
        # Frozen DINO blocks remain in eval mode; selected final blocks train.
        self.backbone.eval()
        for block in self.trainable_blocks:
            block.train(mode)
        self.backbone.norm.train(mode)
        return self

    def forward(self, images, window_valid, window_positions):
        # images: [B, 5, 10, 3, 224, 224], uint8
        # masks/positions: [B, 5, 10]
        if images.ndim != 6:
            raise ValueError(f"Expected 6D images, received {tuple(images.shape)}")
        B, R, W, C, H, WW = images.shape
        expected = (self.n_roles, self.n_windows, 3, self.image_size, self.image_size)
        if (R, W, C, H, WW) != expected:
            raise ValueError(
                f"Unexpected image shape {tuple(images.shape)}; expected [B, {expected}]."
            )
        if window_valid.shape != (B, R, W):
            raise ValueError("window_valid shape does not match images.")
        if window_positions.shape != (B, R, W):
            raise ValueError("window_positions shape does not match images.")
        if not window_valid.bool().any(dim=(1, 2)).all():
            raise ValueError("At least one study in the batch has no valid windows.")

        # Encode only valid windows; do not waste DINOv2 compute on padded slots.
        flat_valid = window_valid.bool().reshape(-1)
        valid_indices = flat_valid.nonzero(as_tuple=False).squeeze(1)
        flat_images = images.reshape(B * R * W, C, H, WW)
        selected = flat_images.index_select(0, valid_indices).float().div_(255.0)
        selected = (selected - self.image_mean) / self.image_std

        encoded_chunks = []
        for start in range(0, selected.shape[0], self.encoder_chunk_size):
            chunk = selected[start:start + self.encoder_chunk_size]
            feat = self.backbone.forward_features(chunk)
            cls_token = feat["x_norm_clstoken"]
            patch_tokens = feat["x_norm_patchtokens"]
            patch_scores = self.patch_attention(patch_tokens).squeeze(-1)
            patch_scores = patch_scores + self.center_bias.unsqueeze(0)
            patch_weights = torch.softmax(patch_scores, dim=1)
            patch_pool = torch.sum(
                patch_tokens * patch_weights.unsqueeze(-1), dim=1
            )
            encoded_chunks.append(
                self.window_projection(torch.cat([cls_token, patch_pool], dim=-1))
            )
        valid_features = torch.cat(encoded_chunks, dim=0)

        flat_features = valid_features.new_zeros((B * R * W, self.embed_dim))
        flat_features = flat_features.index_copy(0, valid_indices, valid_features)
        tokens = flat_features.view(B, R, W, self.embed_dim)

        # Add role, window-order and normalized slice-position information.
        pos = window_positions.to(dtype=tokens.dtype).clamp(0.0, 1.0)
        pos_features = torch.stack(
            [pos, torch.sin(2.0 * math.pi * pos), torch.cos(2.0 * math.pi * pos)],
            dim=-1,
        )
        pos_features = self.position_projection(pos_features)
        order_ids = torch.arange(W, device=images.device)
        tokens = (
            tokens
            + self.role_embedding.to(dtype=tokens.dtype)
            + self.window_embedding(order_ids).view(1, 1, W, self.embed_dim)
            + pos_features
        )

        # Per-role global means: [B, R, D]. Missing roles remain zero.
        valid = window_valid.bool()
        mask_f = valid.unsqueeze(-1).to(tokens.dtype)
        role_counts = mask_f.sum(dim=2).clamp_min(1.0)
        role_global = (tokens * mask_f).sum(dim=2) / role_counts

        # Independent attention over windows WITHIN each role for each target.
        # Scores: [B, T, R, W]; masked softmax safely handles absent roles.
        scores = torch.einsum(
            "brwd,trd->btrw", tokens, self.target_role_queries
        ) / math.sqrt(self.embed_dim)
        scores = scores.masked_fill(
            ~valid[:, None, :, :], torch.finfo(scores.dtype).min
        )
        attention = torch.softmax(scores.float(), dim=-1).to(tokens.dtype)
        attention = attention * valid[:, None, :, :].to(tokens.dtype)
        attention = attention / attention.sum(dim=-1, keepdim=True).clamp_min(1e-6)
        attended_role = torch.einsum("btrw,brwd->btrd", attention, tokens)

        gamma = torch.sigmoid(self.target_role_residual_logits).view(
            1, self.n_targets, self.n_roles, 1
        )
        target_role_features = role_global[:, None, :, :] + gamma * (
            attended_role - role_global[:, None, :, :]
        )
        target_role_features = self.role_feature_norm(target_role_features)

        # Keep missing-role vectors at zero after LayerNorm; the presence flag
        # below explicitly tells the target head that the role was unavailable.
        role_presence = valid.any(dim=-1).to(target_role_features.dtype)  # [B, R]
        target_role_features = target_role_features * role_presence[:, None, :, None]

        # Concatenate five independently pooled role vectors per target.
        target_role_features = target_role_features.reshape(
            B, self.n_targets, self.n_roles * self.embed_dim
        )
        presence_features = role_presence[:, None, :].expand(
            -1, self.n_targets, -1
        )
        classifier_inputs = torch.cat(
            [target_role_features, presence_features], dim=-1
        )

        # Independent nonlinear prediction head for each target.
        logits = torch.cat(
            [
                self.target_classifiers[t](classifier_inputs[:, t, :])
                for t in range(self.n_targets)
            ],
            dim=1,
        )
        return logits
print("FiveRoleDINOv2_2p5D class defined.")


## 8. Load the five fold models from the offline bundle

In [ ]:
# Instantiate source code only; pretrained=False avoids all weight downloads.
# Each best_checkpoint.pt contains the full DINOv2 backbone state as part of model_state.
try:
    backbone_template = torch.hub.load(
        str(DINOV2_REPO_DIR), DINO_HUB_MODEL, source="local", pretrained=False
    )
except Exception as exc:
    raise RuntimeError(
        "Could not instantiate DINOv2 from the bundled local repository. "
        "Internet is disabled; verify the ZIP contains the complete dinov2_repo/ folder."
    ) from exc

if not hasattr(backbone_template, "forward_features") or len(backbone_template.blocks) != 12:
    raise RuntimeError("Bundled source did not create the expected DINOv2 ViT-S/14 architecture.")
backbone_template.eval()

models = []
fold_model_info = []
for fold, checkpoint_path in enumerate(FOLD_CKPTS, start=1):
    try:
        checkpoint = torch.load(checkpoint_path, map_location="cpu", weights_only=False)
    except TypeError:
        checkpoint = torch.load(checkpoint_path, map_location="cpu")

    if int(checkpoint.get("fold", -1)) != fold:
        raise ValueError(f"Checkpoint fold mismatch: expected {fold}, got {checkpoint.get('fold')}.")
    if list(checkpoint.get("targets", [])) != TARGETS:
        raise ValueError(f"Fold {fold}: target order mismatch.")
    if list(checkpoint.get("role_order", [])) != FIVE_ROLE_ORDER:
        raise ValueError(f"Fold {fold}: role order mismatch.")
    if "model_state" not in checkpoint:
        raise KeyError(f"Fold {fold}: model_state missing from checkpoint.")

    fold_backbone = copy.deepcopy(backbone_template)
    fold_model = FiveRoleDINOv2_2p5D(
        backbone=fold_backbone,
        image_size=IMAGE_SIZE,
        backbone_dim=BACKBONE_DIM,
        embed_dim=EMBED_DIM,
        freeze_early_blocks=FREEZE_EARLY_BLOCKS,
        encoder_chunk_size=ENCODER_CHUNK_SIZE,
    ).to(device)
    fold_model.load_state_dict(checkpoint["model_state"], strict=True)
    fold_model.eval()
    models.append(fold_model)
    fold_model_info.append({
        "fold": fold,
        "best_epoch": int(checkpoint.get("best_epoch", -1)),
        "best_val_loss": float(checkpoint.get("best_val_loss", np.nan)),
        "signature": str(checkpoint.get("signature", "")),
    })
    print(
        f"Loaded fold {fold}/5 | best epoch={fold_model_info[-1]['best_epoch']} "
        f"| best val BCE={fold_model_info[-1]['best_val_loss']:.5f}"
    )
    del checkpoint, fold_backbone, fold_model
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

del backbone_template
gc.collect()
print("Loaded ensemble members:", len(models))
print("Parameters per model (M):", round(sum(p.numel() for p in models[0].parameters()) / 1e6, 2))
print("Allocated GPU memory GiB:", round(torch.cuda.memory_allocated() / 1024**3, 2))

## 9. Run 5-fold ensemble inference on test studies

In [ ]:
test_uids = test_df[ID_COL].astype(str).tolist()
N_TEST = len(test_uids)
fold_probabilities = np.full((N_TEST, len(models), len(TARGETS)), 0.5, dtype=np.float32)
role_diagnostics_all = []
study_inference_rows = []
start_time = time.time()

for study_idx, uid in enumerate(test_uids):
    X, valid_mask, positions, diagnostics, n_roles = build_study_windows(uid)
    role_diagnostics_all.extend(diagnostics)

    if X is None:
        # Neutral fallback only when no usable 2.5D window could be decoded.
        study_inference_rows.append({
            ID_COL: uid, "n_available_roles": int(n_roles),
            "status": "all_roles_unusable_fallback_0.5",
        })
    else:
        images_t = torch.from_numpy(X[None]).to(device, non_blocking=True)
        valid_t = torch.from_numpy(valid_mask[None]).to(device, non_blocking=True).bool()
        positions_t = torch.from_numpy(positions[None]).to(device, non_blocking=True)
        with torch.inference_mode():
            for model_idx, model in enumerate(models):
                with torch.autocast(device_type="cuda", dtype=torch.float16, enabled=USE_AMP):
                    logits = model(images_t, valid_t, positions_t)
                probs = torch.sigmoid(logits.float()).squeeze(0).cpu().numpy()
                fold_probabilities[study_idx, model_idx] = probs.astype(np.float32)
        study_inference_rows.append({
            ID_COL: uid, "n_available_roles": int(n_roles), "status": "ok"
        })
        del images_t, valid_t, positions_t

    if (study_idx + 1) % 10 == 0 or study_idx + 1 == N_TEST:
        elapsed_min = (time.time() - start_time) / 60.0
        print(
            f"Inference {study_idx + 1}/{N_TEST} studies | "
            f"elapsed={elapsed_min:.1f} min | usable roles={n_roles}",
            flush=True,
        )

ensemble_probabilities = fold_probabilities.mean(axis=1)
if ensemble_probabilities.shape != (N_TEST, len(TARGETS)):
    raise RuntimeError(f"Unexpected ensemble shape: {ensemble_probabilities.shape}")
if not np.isfinite(ensemble_probabilities).all():
    raise RuntimeError("Ensemble predictions contain NaN/Inf.")

role_diagnostics_df = pd.DataFrame(role_diagnostics_all)
study_inference_df = pd.DataFrame(study_inference_rows)
role_diagnostics_df.to_csv(OUTPUT_ROOT / "test_role_preprocessing_log.csv", index=False)
study_inference_df.to_csv(OUTPUT_ROOT / "test_study_inference_log.csv", index=False)
print("Inference finished in minutes:", round((time.time() - start_time) / 60.0, 2))
print("Studies without usable role windows:", int(study_inference_df["status"].ne("ok").sum()))
print("Mean available roles/study:", round(float(study_inference_df["n_available_roles"].mean()), 3))
print("Ensemble probability range:", float(ensemble_probabilities.min()), "to", float(ensemble_probabilities.max()))

## 10. Build and validate `submission.csv`

In [ ]:
probability_df = pd.DataFrame(ensemble_probabilities, columns=TARGETS)
probability_df.insert(0, ID_COL, test_uids)

submission = sample_submission[[ID_COL]].merge(
    probability_df, on=ID_COL, how="left", validate="one_to_one"
)
submission = submission[EXPECTED_COLUMNS]

if len(submission) != len(sample_submission):
    raise ValueError("Submission row count differs from sample_submission.")
if submission[ID_COL].tolist() != sample_submission[ID_COL].tolist():
    raise ValueError("Submission StudyInstanceUID order differs from sample_submission.")
if submission[EXPECTED_COLUMNS].isna().any().any():
    raise ValueError("Submission has missing values after UID merge.")
values = submission[TARGETS].to_numpy(dtype=np.float64)
if not np.isfinite(values).all():
    raise ValueError("Submission contains non-finite scores.")
if (values < 0).any() or (values > 1).any():
    raise ValueError("Submission probabilities must lie in [0,1].")

SUBMISSION_PATH = Path("/kaggle/working/submission.csv")
submission.to_csv(SUBMISSION_PATH, index=False, float_format="%.8f")

fold_prediction_rows = []
for i, uid in enumerate(test_uids):
    for fold_idx in range(len(models)):
        row = {ID_COL: uid, "fold": fold_idx + 1}
        row.update({target: float(fold_probabilities[i, fold_idx, j]) for j, target in enumerate(TARGETS)})
        fold_prediction_rows.append(row)
pd.DataFrame(fold_prediction_rows).to_csv(OUTPUT_ROOT / "test_predictions_by_fold.csv", index=False)

submission_manifest = {
    "architecture": bundle_manifest.get("architecture"),
    "checkpoint_bundle": str(BUNDLE_PATH),
    "fold_models": fold_model_info,
    "ensemble": "arithmetic mean of five fold sigmoid probabilities",
    "competition_root": str(COMP_ROOT),
    "test_series_root": str(TEST_SERIES_DIR),
    "n_test_studies": int(N_TEST),
    "n_studies_with_usable_windows": int(study_inference_df["status"].eq("ok").sum()),
    "n_studies_fallback_0_5": int(study_inference_df["status"].ne("ok").sum()),
    "targets": TARGETS,
    "role_order": FIVE_ROLE_ORDER,
    "preprocessing": {
        "slices_per_role": N_ROLE_SLICES,
        "windows_per_role": N_ROLE_WINDOWS,
        "window_channels": 3,
        "image_size": IMAGE_SIZE,
        "slice_sampling": "12 uniformly spaced slices; fill failed samples from remaining geometrically sorted slices",
        "slice_sorting": "projected ImagePositionPatient; fallback InstanceNumber; fallback natural filename",
        "intensity_normalization": "per-series percentile [1.0, 99.5] across decoded sampled slices",
        "resize": "OpenCV INTER_AREA to 224x224",
    },
    "submission_metric": "macro-average ROC-AUC across 12 targets",
    "submission_file": str(SUBMISSION_PATH),
}
(OUTPUT_ROOT / "submission_manifest.json").write_text(
    json.dumps(submission_manifest, ensure_ascii=False, indent=2), encoding="utf-8"
)

print("========== SUBMISSION VALIDATION PASS ==========")
print("File:", SUBMISSION_PATH)
print("Shape:", submission.shape)
print("Columns:", submission.columns.tolist())
print("Prediction min/max:", float(values.min()), float(values.max()))
print("Fallback studies:", int(study_inference_df["status"].ne("ok").sum()))
display(submission.head())

## 11. Final Kaggle output check

In [ ]:
FINAL_SUBMISSION_PATH = Path("/kaggle/working/submission.csv")
if not FINAL_SUBMISSION_PATH.is_file():
    raise FileNotFoundError("submission.csv was not created.")
check = pd.read_csv(FINAL_SUBMISSION_PATH, dtype={ID_COL: str})
if check.shape != sample_submission.shape:
    raise RuntimeError(f"Saved CSV shape mismatch: {check.shape} vs {sample_submission.shape}")
if check.columns.tolist() != EXPECTED_COLUMNS:
    raise RuntimeError("Saved CSV column order does not match sample_submission.")
if check[ID_COL].tolist() != sample_submission[ID_COL].tolist():
    raise RuntimeError("Saved CSV UID order does not match sample_submission.")
if check.isna().any().any():
    raise RuntimeError("Saved CSV contains missing values.")
check_values = check[TARGETS].to_numpy(dtype=np.float64)
if not np.isfinite(check_values).all() or (check_values < 0).any() or (check_values > 1).any():
    raise RuntimeError("Saved CSV contains invalid prediction values.")

print("KAGGLE SUBMISSION OUTPUT: PASS")
print("File:", FINAL_SUBMISSION_PATH)
print("Rows:", len(check))
print("Columns:", len(check.columns))
print("Size (bytes):", FINAL_SUBMISSION_PATH.stat().st_size)
print("Internet requirement: OFF (local bundled DINOv2 source + checkpoint weights).")
display(check.head())